In [ ]:
# =====================================================
# 0. 라이브러리 / 파일 설정
# =====================================================

import re
import json
import time
import pandas as pd
from tqdm import tqdm

# 이미 OpenAI client를 만들었다면 생략
from openai import OpenAI
client = OpenAI()
 
INPUT_FILE = "chelsea_reanalyze_target.xlsx"

OUTPUT_XLSX = "chelsea_reanalyze_target_labeled.xlsx"
OUTPUT_CSV = "chelsea_reanalyze_target_labeled.csv"


# =====================================================
# 1. GPT 응답 JSON 파싱 함수
# =====================================================

def parse_json_array(text):
    text = text.strip()

    # 코드블록 제거
    text = re.sub(r"^```json|^```|```$", "", text, flags=re.MULTILINE).strip()

    # JSON 배열 부분만 추출
    start = text.find("[")
    end = text.rfind("]")

    if start != -1 and end != -1:
        text = text[start:end+1]

    return json.loads(text)


# =====================================================
# 2. 데이터 불러오기
# =====================================================

df = pd.read_excel(INPUT_FILE)

df = df.reset_index(drop=True).copy()
df["analysis_id"] = df.index

df["text_clean"] = df["text_clean"].fillna("").astype(str)
df = df[df["text_clean"].str.len() >= 2].reset_index(drop=True)

print("분석 대상 데이터 수:", len(df))
print("\n[source 분포]")
print(df["source"].value_counts(dropna=False))

print("\n[content_type 분포]")
print(df["content_type"].value_counts(dropna=False))

print("\n[gpt_sentiment 기존 분포]")
print(df["gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 3. match_context 생성용 라운드별 경기 맥락
# =====================================================
# 여기에 첼시 29R~38R 경기 결과/맥락을 직접 입력

match_context_map = {
    "29R": "첼시가 아스톤빌라를 상대로 4:1로 승리",
    "30R": "첼시가 뉴캐슬에게 1:0으로 패배",
    "31R": "첼시가 에버튼에게 3:0으로 패배",
    "32R": "첼시가 맨시티에게 3:0으로 패배",
    "33R": "첼시가 맨유에게 1:0으로 패배",
    "34R": "첼시가 브라이튼에게 3:0으로 패배",
    "35R": "첼시가 노팅엄 포레스트에게 3:1로 패배",
    "36R": "첼시가 리버풀과 1:1 무승부",
    "37R": "첼시가 토트넘을 상대로 2:1로 승리",
    "38R": "첼시가 선덜랜드에게 2:1로 패배 후 유럽 대항전 진출 실패",
}

In [ ]:
# =====================================================
# 4. match_context 생성 함수
# =====================================================

def make_match_context(row):
    source = row.get("source")
    round_info = row.get("round")
    title = row.get("title")
    period = row.get("period")
    content_type = row.get("content_type")
    content_published_at = row.get("content_published_at")
    comment_published_at = row.get("comment_published_at")

    # 유튜브: 라운드별 경기 맥락 사용
    if source == "youtube":
        base_context = match_context_map.get(
            round_info,
            "첼시 관련 유튜브 영상 댓글. 영상 제목과 댓글 내용을 바탕으로 판단."
        )

        return f"""
{base_context}

[데이터 정보]
데이터 출처: YouTube 댓글
영상 제목: {title}
라운드: {round_info}
영상 업로드일: {content_published_at}
댓글 작성일: {comment_published_at}
38R 기준 구분: {period}

이 댓글은 첼시 관련 유튜브 영상에 달린 댓글이다.
영상 제목과 라운드 정보를 댓글 해석에 반영한다.
"""

    # 카페: 경기 하이라이트가 아니라 팬 커뮤니티 의견
    if source == "cafe":
        if period == "before_38R":
            return f"""
[데이터 정보]
데이터 출처: 첼시 팬 카페
게시글 제목: {title}
데이터 유형: {content_type}
게시글 작성일: {content_published_at}
댓글 작성일: {comment_published_at}
38R 기준 구분: before_38R

이 텍스트는 38R 이전 첼시 팬 카페 게시글 또는 댓글이다.
시즌 종료 직전 팬들의 기대, 우려, 경기 전 전망, 선수단 평가, 감독/전술 평가가 포함될 수 있다.
영상 장면에 대한 즉각 반응이 아니라 팬 커뮤니티의 의견으로 해석한다.
"""
        elif period == "after_38R":
            return f"""
[데이터 정보]
데이터 출처: 첼시 팬 카페
게시글 제목: {title}
데이터 유형: {content_type}
게시글 작성일: {content_published_at}
댓글 작성일: {comment_published_at}
38R 기준 구분: after_38R

이 텍스트는 38R 이후 첼시 팬 카페 게시글 또는 댓글이다.
시즌 종료 직후 팬들의 시즌 총평, 선수 평가, 감독 평가, 이적시장 의견, 구단 운영 평가가 포함될 수 있다.
영상 장면에 대한 즉각 반응이 아니라 팬 커뮤니티의 의견으로 해석한다.
"""
        else:
            return f"""
[데이터 정보]
데이터 출처: 첼시 팬 카페
게시글 제목: {title}
데이터 유형: {content_type}

이 텍스트는 첼시 팬 카페 게시글 또는 댓글이다.
팬 커뮤니티의 의견으로 해석한다.
"""

    return "첼시 관련 텍스트 데이터."


df["match_context"] = df.apply(make_match_context, axis=1)

# 같은 맥락끼리 묶기 위한 key
df["context_key"] = (
    df["source"].astype(str) + " | " +
    df["round"].astype(str) + " | " +
    df["period"].astype(str) + " | " +
    df["title"].astype(str)
)

print("match_context 결측 수:", df["match_context"].isna().sum())

display(
    df[[
        "source",
        "round",
        "period",
        "title",
        "content_type",
        "text_clean",
        "match_context"
    ]].head()
)

In [ ]:
# =====================================================
# 5. 1차 감정분석 함수
# =====================================================

def analyze_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 한국어 축구 팬덤 댓글을 분석하는 target-aware 감정분석 분류기다.
분석 대상은 EPL 첼시 관련 유튜브, 카페 댓글이다.

[첼시 선수단 정보]
현재 감독: 맥팔레인(임시 감독)
이전 감독: 로세니어, 마레스카

선수:
로베르트 산체스, 쿠쿠레야(쿠쿠렐라), 토신(토신 아다라비요), 바디아실, 콜윌, 페드루 네트, 엔소 페르난데스,
리암 델랍, 콜 파머, 제이미 기튼스, 요르겐센, 다리우 이수구, 안드레이 산투스, 마마두 사르, 주앙 페드루,
조렐 하토, 트레보 찰로바, 리스 제임스(리제), 카이세도, 귀스토(구스토), 포파나, 아체암퐁, 마르크 기우, 
이스테방, 라비아, 가르나초

[선수단 반영 규칙]
- 위 선수나 감독을 칭찬하면 첼시 관련 positive로 분류한다.
- 위 선수나 감독을 비판, 조롱, 실망하면 첼시 관련 negative로 분류한다.
- 위 선수 이름이 직접 나오면 target은 보통 chelsea_player로 둔다.
- 맥팔레인, 로세니어, 마레스카 관련 댓글이면 target은 chelsea_manager로 둔다.
- 첼시 이외의 다른 대상을 언급한 댓글은 첼시 감정으로 보지 않고 neutral로 분류한다.

[영상/게시글/경기 맥락]
{match_context}

이 맥락은 댓글 해석에 반드시 반영한다.
첼시가 승리한 경기에서는 "골 미쳤다", "마무리 좋다", "슈팅 지렸다", "수비 좋다"처럼 대상이 불명확한 경기력 칭찬은 특별히 상대팀 언급이 없으면 첼시 positive로 우선 판단한다.
첼시가 패배한 경기에서는 아쉬움, 비판, 조롱, 실망 표현이 첼시를 향하면 negative로 판단한다.

[분류 기준]
- positive: 첼시 팀/선수/감독/전술/경기력/결과에 대한 칭찬, 감탄, 기대, 축하
- negative: 첼시 팀/선수/감독/전술/경기력/결과에 대한 비판, 실망, 조롱, 비꼼, 분노
- neutral: 첼시와 직접 관련 없거나 단순 질문/정보/감정 판단 불가

[중요 규칙]
- neutral은 정말 감정 방향을 판단할 수 없을 때만 사용한다.
- 첼시 관련 맥락이 있고 긍정/부정 뉘앙스가 조금이라도 있으면 positive 또는 negative로 판단한다.
- "미쳤다", "지린다", "개쩐다", "폼 미쳤다", "찢었다", "월클", "GOAT"는 칭찬 맥락이면 positive.
- "개못한다", "답답하다", "망했다", "처참하다", "노답", "왜 저러냐"는 비판 맥락이면 negative.
- 상대팀/상대선수 칭찬은 첼시와 무관하면 neutral.
- 단, 상대팀 칭찬이 첼시의 패배/부진/약점을 강조하면 negative.
- 단순 감탄사라도 댓글 맥락상 첼시 경기 장면에 대한 반응으로 해석 가능하면 neutral보다 positive/negative를 우선한다.

[target 기준]
- chelsea_team
- chelsea_player
- chelsea_manager
- opponent
- referee_or_league
- unclear

[스코어/라운드 맥락 활용]
라운드 정보와 스코어는 댓글 해석에 반영한다.
match_context에서 상대팀이 득점하지 못한 경기라고 주어진 경우,
"골 미쳤다", "마무리 좋다", "슈팅 지렸다", "득점 장면 좋다"처럼 득점/공격 장면을 칭찬하는 댓글은
특별히 상대팀이나 상대 선수가 명시되지 않는 한 첼시 positive로 판단한다.

첼시가 패배한 경기에서는 경기력 비판, 실망, 조롱, 비꼼 표현을 negative로 판단한다.
단, 패배 경기라도 첼시 선수 개인의 좋은 플레이를 칭찬하면 positive로 판단한다.

[출력 규칙]
반드시 JSON 배열만 출력하라.
설명 문장, 코드블록, 마크다운은 쓰지 마라.
각 결과에는 입력 번호 id를 포함하라.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0부터 1.0 사이 숫자,
    "target": "chelsea_team 또는 chelsea_player 또는 chelsea_manager 또는 opponent 또는 referee_or_league 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[분석할 댓글]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {"role": "system", "content": "너는 한국어 축구 댓글 감정분석 전문가다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "error",
                "confidence": 0.0,
                "target": "error",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]

In [ ]:
# =====================================================
# 6. 2차 neutral 재분석 함수
# =====================================================

def analyze_neutral_batch(comments, match_context):
    comments_text = "\n".join([
        f"{i}. {comment}"
        for i, comment in enumerate(comments)
    ])

    prompt = f"""
너는 첼시 관련 유튜브, 카페 댓글의 2차 감정분석 분류기다.
아래 댓글들은 1차 분석에서 neutral로 분류되었지만, neutral 판정을 최소화하기 위해 재검토한다.

[맥락]
{match_context}

[가장 중요한 원칙]
neutral은 예외값이다.
감정 방향이 정말 전혀 없을 때만 neutral로 둔다.

칭찬, 감탄, 만족, 축하, 기대가 조금이라도 있으면 positive.
비판, 실망, 조롱, 분노, 답답함, 비꼼이 조금이라도 있으면 negative.

댓글이 짧거나 대상이 생략되어도,
경기 맥락상 첼시에 대한 반응으로 자연스럽게 해석되면
positive 또는 negative로 분류한다.

[neutral로 유지하는 경우]
- 단순 질문
- 경기 시간, 일정, 정보 확인
- 영상 편집, 중계, 해설, 업로드 관련 이야기
- 첼시 / 첼시 선수 / 감독 / 경기 결과와 감정적으로 연결되지 않는 말
- 감정 단서가 전혀 없는 단순 사실 전달

[neutral 금지 예시]
"미쳤다", "지렸다", "좋다", "깔끔하다", "나이스", "레전드", "GOAT" → positive
"답답하다", "망했다", "왜 저러냐", "최악", "또 이러네", "아쉽다" → negative

"역시 첼시"은 칭찬이면 positive, 조롱/비꼼이면 negative.
상대팀 칭찬이라도 첼시의 패배, 약점, 부진을 강조하면 negative.

[첼시 팬덤 멸칭 및 조롱 표현 규칙]

다음 표현은 첼시를 조롱하거나 비하하는 표현으로 간주한다.

아래 단어 또는 유사 표현이 포함되면 sentiment를 negative로 분류한다.

* 첼강딱
* 돈시
* 첼레반
* 첼시놈
* 보엘리 FC
* 1조 FC
* 돈지랄 FC
* 강등시
* 돈은 돈대로 쓰고
* 돈 처바르고
* 천억 쓰고
* 1조 쓰고

예시

* 첼강딱 수준
* 돈시 또 시작이네
* 1조 쓰고 이게 맞냐
* 보엘리 FC 레전드
* 돈은 돈대로 쓰고 경기력은 최악

위 표현들은 첼시에 대한 조롱 및 비하 표현으로 간주하며 negative로 분류한다.

[조롱성 웃음 표현 규칙]

댓글에 "ㅋㅋ", "ㅋㅋㅋ", "개웃기네", "존나 웃기네", "웃음벨", "개꿀잼", "레전드", "과학" 등이 포함되어 있더라도

첼시 또는 첼시 선수의 실수, 패배, 골결정력, 부진, 실책을 비웃는 맥락이면 반드시 negative로 분류한다.

예시:

- 또 지랄하네 ㅋㅋㅋ
- 첼시는 과학이네
- 파머 폼 다 죽었네 ㅋㅋ
- 첼시 또 졌네 개웃기네

[판단 우선순위]
1. 비판 / 조롱 / 실망 단서가 있으면 negative
2. 칭찬 / 감탄 / 만족 단서가 있으면 positive
3. 둘 다 전혀 없을 때만 neutral

[출력 규칙]
반드시 JSON 배열만 출력한다.
설명 문장, 코드블록, 마크다운은 출력하지 않는다.

[
  {{
    "id": 0,
    "sentiment": "positive 또는 negative 또는 neutral",
    "confidence": 0.0,
    "target": " chelsea_team 또는 chelsea_player 또는 chelsea_manager 또는 opponent 또는 referee_or_league 또는 unclear",
    "reason": "짧은 한국어 이유"
  }}
]

[재분석할 댓글]
{comments_text}
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[
                {
                    "role": "system",
                    "content": "너는 한국어 축구 댓글 감정분석 전문가다. neutral 판정은 매우 보수적으로 사용한다."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        result_text = response.choices[0].message.content.strip()
        return parse_json_array(result_text)

    except Exception as e:
        return [
            {
                "id": i,
                "sentiment": "neutral",
                "confidence": 0.0,
                "target": "unclear",
                "reason": str(e)
            }
            for i in range(len(comments))
        ]

In [ ]:
# =====================================================
# 7. 1차 감정분석 실행
# =====================================================

def run_first_pass(df, batch_size=20):
    final_df = df.copy()

    final_df["new_gpt_sentiment"] = None
    final_df["new_gpt_confidence"] = None
    final_df["new_gpt_target"] = None
    final_df["new_gpt_reason"] = None

    for context_key, group_df in tqdm(final_df.groupby("context_key"), desc="1차 감정분석"):
        match_context = group_df["match_context"].iloc[0]

        group_df = group_df.reset_index()

        for start in range(0, len(group_df), batch_size):
            batch_df = group_df.iloc[start:start+batch_size]
            comments = batch_df["text_clean"].tolist()

            batch_results = analyze_batch(comments, match_context)

            result_map = {}

            for r in batch_results:
                try:
                    result_map[int(r.get("id"))] = r
                except:
                    pass

            for local_id, row in enumerate(batch_df.itertuples()):
                r = result_map.get(local_id, {})

                idx = row.index

                sentiment = str(r.get("sentiment", "error")).lower().strip()

                if sentiment not in ["positive", "negative", "neutral", "error"]:
                    sentiment = "error"

                final_df.at[idx, "new_gpt_sentiment"] = sentiment
                final_df.at[idx, "new_gpt_confidence"] = r.get("confidence", 0.0)
                final_df.at[idx, "new_gpt_target"] = r.get("target", "unclear")
                final_df.at[idx, "new_gpt_reason"] = r.get("reason", "")

            time.sleep(0.3)

    return final_df


labeled_df = run_first_pass(df, batch_size=20)

print("1차 감정분석 결과")
print(labeled_df["new_gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 8. 2차 neutral 재분석
# =====================================================

def run_second_pass_for_neutral(labeled_df, batch_size=20):
    final_df = labeled_df.copy()

    final_df["gpt_sentiment_1st"] = final_df["new_gpt_sentiment"]
    final_df["gpt_reason_1st"] = final_df["new_gpt_reason"]
    final_df["neutral_second_pass"] = False

    neutral_df = final_df[
        final_df["new_gpt_sentiment"].astype(str).str.lower().eq("neutral")
    ].copy()

    print("2차 재분석 대상 neutral 수:", len(neutral_df))

    for context_key, group_df in tqdm(neutral_df.groupby("context_key"), desc="2차 neutral 재분석"):
        match_context = group_df["match_context"].iloc[0]

        group_df = group_df.reset_index()

        for start in range(0, len(group_df), batch_size):
            batch_df = group_df.iloc[start:start+batch_size]
            comments = batch_df["text_clean"].tolist()

            batch_results = analyze_neutral_batch(comments, match_context)

            result_map = {}

            for r in batch_results:
                try:
                    result_map[int(r.get("id"))] = r
                except:
                    pass

            for local_id, row in enumerate(batch_df.itertuples()):
                r = result_map.get(local_id, {})

                idx = row.index

                sentiment = str(r.get("sentiment", "neutral")).lower().strip()

                if sentiment not in ["positive", "negative", "neutral"]:
                    sentiment = "neutral"

                final_df.at[idx, "new_gpt_sentiment"] = sentiment
                final_df.at[idx, "new_gpt_confidence"] = r.get("confidence", 0.0)
                final_df.at[idx, "new_gpt_target"] = r.get("target", "unclear")
                final_df.at[idx, "new_gpt_reason"] = "[2차 neutral 재분석] " + str(r.get("reason", ""))
                final_df.at[idx, "neutral_second_pass"] = True

            time.sleep(0.3)

    return final_df


labeled_df = run_second_pass_for_neutral(labeled_df, batch_size=20)

print("2차 재분석 후 결과")
print(labeled_df["new_gpt_sentiment"].value_counts(dropna=False))

In [ ]:
# =====================================================
# 9. 기존 gpt 컬럼 갱신
# =====================================================

labeled_df["gpt_sentiment_old"] = labeled_df["gpt_sentiment"]
labeled_df["gpt_confidence_old"] = labeled_df["gpt_confidence"]
labeled_df["gpt_target_old"] = labeled_df["gpt_target"]
labeled_df["gpt_reason_old"] = labeled_df["gpt_reason"]

labeled_df["gpt_sentiment"] = labeled_df["new_gpt_sentiment"]
labeled_df["gpt_confidence"] = labeled_df["new_gpt_confidence"]
labeled_df["gpt_target"] = labeled_df["new_gpt_target"]
labeled_df["gpt_reason"] = labeled_df["new_gpt_reason"]


# =====================================================
# 10. 저장
# =====================================================

date_cols = [
    "content_published_at",
    "comment_published_at"
]

labeled_df_excel = labeled_df.copy()

for col in date_cols:
    if col in labeled_df_excel.columns:
        labeled_df_excel[col] = pd.to_datetime(
            labeled_df_excel[col],
            errors="coerce",
            utc=True
        ).dt.tz_convert(None)

labeled_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

labeled_df_excel.to_excel(
    OUTPUT_XLSX,
    index=False
)

print("✅ 저장 완료")
print("CSV:", OUTPUT_CSV)
print("Excel:", OUTPUT_XLSX)


# =====================================================
# 11. 결과 미리보기
# =====================================================

display(
    labeled_df[
        [
            "source",
            "round",
            "period",
            "title",
            "content_type",
            "text_clean",
            "gpt_sentiment_1st",
            "gpt_sentiment",
            "gpt_confidence",
            "gpt_target",
            "gpt_reason",
            "neutral_second_pass"
        ]
    ].head(20)
)